In [ ]:
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import DataLoader
DATASET = 'sviridov/nbadeepfm'


dataset = load_dataset(DATASET)

combined_dataset = concatenate_datasets(list(dataset.values()))
combined_dataset = combined_dataset.with_format('torch')


/home/sviridov/analytics/nba-deepfm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[Errno -2] Name or service not known' thrown while requesting HEAD https://huggingface.co/datasets/sviridov/nbadeepfm/resolve/74c7b2048eca5e35f89c8c6cf2636f9d197699fc/nbadeepfm.py
Retrying in 1s [Retry 1/5].
Using the latest cached version of the dataset since sviridov/nbadeepfm couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at /home/sviridov/.cache/huggingface/datasets/sviridov___nbadeepfm/default/0.0.0/74c7b2048eca5e35f89c8c6cf2636f9d197699fc (last modified on Mon Mar 16 01:50:54 2026).


5347584  training shot cycles


In [ ]:
import polars as pl
import torch


players_df = pl.read_csv('csv/player.csv')

all_ids = players_df['id'].unique()
null_player_id = 0
nba_id_to_idx_mapping = {}
for idx, nba_id in enumerate(all_ids):
    nba_id_to_idx_mapping[nba_id] = idx + 1 # reserve 0 for null player id

nba_id_to_idx_mapping[null_player_id] = 0

max_nba_id = max(nba_id_to_idx_mapping.keys())
lookup_tensor = torch.zeros(max_nba_id + 1, dtype=torch.long)
for nba_id, sequential_idx in nba_id_to_idx_mapping.items():
    lookup_tensor[nba_id] = sequential_idx
    

In [ ]:
# use external variable unfortunately
def preprocess_function(dataset: DataLoader):
    # 1. Combine players into a single lineup token sequence [Batch, 10]
    lineups = torch.stack([
        lookup_tensor[dataset['offensive_players']],
        lookup_tensor[dataset['defensive_players']]
    ], dim=1).view(-1, 10)


    # 2. Extract Role IDs (Shooter, Assister, Rebounder, Defender)
    # Ensure these are handled as a single [Batch, 4] tensor

    shooting_player = dataset['shooting_player'].nan_to_num(0).to(torch.long)
    assisting_player = dataset['assisting_player'].nan_to_num(0).to(torch.long)
    rebounding_player = dataset['rebounding_player'].nan_to_num(0).to(torch.long)
    defending_player = dataset['defending_player'].nan_to_num(0).to(torch.long)

    roles = torch.stack([
        lookup_tensor[shooting_player],
        lookup_tensor[assisting_player],
        lookup_tensor[rebounding_player],
        lookup_tensor[defending_player]
    ], dim=1)

    # 3. Pack Privileged Info (PI) into a single [Batch, 6] float tensor
    pi_stats = torch.stack([
        dataset['shot_distance'].nan_to_num(0.0),
        dataset['is_putback'].to(torch.float32),
        dataset['is_and1'].to(torch.float32),
        dataset['is_freethrow'].to(torch.float32),
        dataset['is_turnover'].to(torch.float32),
        dataset['is_steal'].to(torch.float32)
    ], dim=1)

    return {
        "lineup_ids": lineups,
        "role_ids": roles,
        "pi_stats": pi_stats,
        "labels": dataset['shot_points_scored'].nan_to_num(0).to(torch.long)
    }

# Apply mapping (Fast and memory-efficient)
processed_dataset = combined_dataset.map(
    preprocess_function, 
    batched=True, 
    remove_columns=combined_dataset.column_names
)


processed_dataset.set_format("torch")

loader = DataLoader(
    processed_dataset, 
    batch_size=2048,      # Critical for GPU utilization
    shuffle=True, 
    num_workers=4,         
    pin_memory=True,       
    persistent_workers=True # Keeps data loading alive between epochs
)


print(len(loader) * 2048, " training shot cycles")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
from NBADeepFm import ShotCycle, NBATransformer

model = NBATransformer(num_players=len(nba_id_to_idx_mapping), embedding_dim=32)
model = model.to(device)


In [ ]:
# train the model
from torch.amp import autocast, GradScaler

scaler = GradScaler()
criterion = nn.CrossEntropyLoss().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5).to(device)

num_epochs = 1
batch_size = 64

for epoch in range(num_epochs):
    for i, batch in enumerate(loader):
        optimizer.zero_grad()
        inputs = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        labels = inputs.pop("labels")

        with autocast():
        # Your Teacher/Student model logic
            logits = model(**inputs) 
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        if (i+1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(dataset)}], Loss: {loss.item():.4f}')
    
    torch.save(model, f"nba_transformer_epoch{epoch}_checkpoint.pth")
            


print("Finished training model")

In [ ]:
#upload to hugging face hub


# model.push_to_hub("sviridov/nba-transformer", private=True)